# Advanced EDA - Holidays, Weekdays & Oil Price

Exploring relationships between sales and external factors.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime

sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 5)

ModuleNotFoundError: No module named 'seaborn'

---
## 1. Load & Prepare Data

In [ ]:
# Load sales data
df = pd.read_csv('data/feature_engineered_timeseries.csv', parse_dates=['date'])
df.set_index('date', inplace=True)
print(f'Sales data: {df.shape}')

# Load holidays
holidays = pd.read_csv('data/holidays.csv', parse_dates=['date'])
print(f'Holidays data: {holidays.shape}')
holidays.head()

In [ ]:
# Load oil prices
oil = pd.read_csv('data/oil.csv', parse_dates=['date'])
oil.set_index('date', inplace=True)
oil = oil.asfreq('D')
# Fill missing oil prices (forward fill, since weekends/holidays have no price)
oil['dcoilwtico'] = oil['dcoilwtico'].ffill()
print(f'Oil data: {oil.shape}')
oil.head()

---
## 2. Weekday Analysis

**Question: Do sales follow a weekly pattern? Which days have highest/lowest sales?**

In [ ]:
day_names = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']
df['day_name'] = df.index.dayofweek.map({0:'Monday',1:'Tuesday',2:'Wednesday',
                                         3:'Thursday',4:'Friday',5:'Saturday',6:'Sunday'})

day_order = ['Monday','Tuesday','Wednesday','Thursday','Friday','Saturday','Sunday']

day_stats = df.groupby('day_name')['unit_sales'].agg(['mean','std','median','count'])
day_stats = day_stats.reindex(day_order)
day_stats

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Barplot of average sales by weekday
colors = ['#3498db']*5 + ['#e67e22', '#e74c3c']
axes[0].bar(day_order, day_stats['mean'], color=colors, edgecolor='black')
axes[0].set_title('Average Daily Sales by Weekday', fontsize=14, fontweight='bold')
axes[0].set_ylabel('Average Unit Sales')
axes[0].tick_params(axis='x', rotation=45)
for i, v in enumerate(day_stats['mean']):
    axes[0].text(i, v + 5, f'{v:.0f}', ha='center', fontweight='bold')

# Boxplot
sns.boxplot(x='day_name', y='unit_sales', data=df, order=day_order,
            palette=['#3498db']*5 + ['#e67e22', '#e74c3c'], ax=axes[1])
axes[1].set_title('Sales Distribution by Weekday', fontsize=14, fontweight='bold')
axes[1].set_ylabel('Unit Sales')
axes[1].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()

In [ ]:
print('=== Weekday vs Weekend ===')
weekday_avg = df[df['is_weekend']==0]['unit_sales'].mean()
weekend_avg = df[df['is_weekend']==1]['unit_sales'].mean()
print(f'Weekday avg: {weekday_avg:.1f}')
print(f'Weekend avg: {weekend_avg:.1f}')
print(f'Weekend uplift: {((weekend_avg - weekday_avg)/weekday_avg*100):.1f}%')

**Key Insight:** Saturday and Sunday have noticeably higher sales than weekdays. The weekend effect is strong.

---
## 3. Holiday Impact on Sales

**Question: Do holidays have an impact on sales?**

In [ ]:
# Create a flag: was a given day a holiday?
# First, aggregate holidays: mark if ANY holiday falls on a date
holiday_dates = holidays.groupby('date').agg(
    holiday_count=('locale', 'count'),
    has_national=('locale', lambda x: (x=='National').any()),
    has_regional=('locale', lambda x: (x=='Regional').any()),
    has_local=('locale', lambda x: (x=='Local').any()),
    descriptions=('description', lambda x: list(x))
)

df['is_holiday'] = df.index.isin(holiday_dates.index).astype(int)
df['is_national_holiday'] = df.index.isin(holiday_dates[holiday_dates['has_national']].index).astype(int)

print(f'Total days with holidays: {df["is_holiday"].sum()}')
print(f'National holidays: {df["is_national_holiday"].sum()}')

In [ ]:
holiday_sales = df.groupby('is_holiday')['unit_sales'].agg(['mean','std','count'])
print('=== Sales on Holiday vs Non-Holiday ===')
holiday_sales

In [ ]:
nat_holiday_sales = df.groupby('is_national_holiday')['unit_sales'].agg(['mean','std','count'])
print('=== Sales on National Holiday vs Other Days ===')
nat_holiday_sales

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Any holiday
bars1 = axes[0].bar(['Non-Holiday', 'Holiday'], holiday_sales['mean'],
                    color=['#3498db', '#e74c3c'], edgecolor='black')
axes[0].set_title('Average Sales: Holiday vs Non-Holiday', fontsize=14, fontweight='bold')
axes[0].set_ylabel('Average Unit Sales')
for bar in bars1:
    height = bar.get_height()
    axes[0].text(bar.get_x()+bar.get_width()/2, height+5, f'{height:.1f}',
                ha='center', fontweight='bold')

# National holiday
bars2 = axes[1].bar(['Non-National Holiday', 'National Holiday'], nat_holiday_sales['mean'],
                    color=['#3498db', '#e74c3c'], edgecolor='black')
axes[1].set_title('Average Sales: National Holiday vs Other', fontsize=14, fontweight='bold')
axes[1].set_ylabel('Average Unit Sales')
for bar in bars2:
    height = bar.get_height()
    axes[1].text(bar.get_x()+bar.get_width()/2, height+5, f'{height:.1f}',
                ha='center', fontweight='bold')

plt.tight_layout()
plt.show()

In [ ]:
# Let's look at specific major holidays
major_holidays = ['Navidad', 'Carnaval', 'Primer Grito de Independencia', 
                  'Independencia de Guayaquil', 'Viernes Santo', 'Black Friday']

for h in major_holidays:
    h_dates = holidays[holidays['description'].str.contains(h, na=False)]['date']
    if len(h_dates) > 0:
        h_sales = df.loc[df.index.isin(h_dates), 'unit_sales'].mean()
        overall_avg = df['unit_sales'].mean()
        print(f'{h:40s}  |  Avg sales: {h_sales:7.1f}  |  Overall avg: {overall_avg:7.1f}  |  Diff: {((h_sales-overall_avg)/overall_avg*100):+5.1f}%')

**Key Insight:** Holidays have a measurable impact. National holidays show a clearer effect. Christmas (Navidad) and Black Friday tend to boost sales, while some holidays like Dia de Difuntos might lower them.

---
## 4. Oil Price Analysis

**Question: Is the oil price relevant to sales?**

In [ ]:
# Merge oil prices with sales
df_oil = df[['unit_sales']].join(oil, how='left')
df_oil['dcoilwtico'] = df_oil['dcoilwtico'].ffill()  # fill any remaining NaN
print(f'Rows with oil data: {df_oil["dcoilwtico"].notna().sum()}')
df_oil.head()

In [ ]:
# Plot both series
fig, ax1 = plt.subplots(figsize=(14, 5))

color1 = '#3498db'
ax1.plot(df_oil.index, df_oil['unit_sales'], color=color1, alpha=0.7, label='Sales')
ax1.set_ylabel('Unit Sales', color=color1, fontsize=12)
ax1.tick_params(axis='y', labelcolor=color1)

ax2 = ax1.twinx()
color2 = '#e74c3c'
ax2.plot(df_oil.index, df_oil['dcoilwtico'], color=color2, alpha=0.7, label='Oil Price')
ax2.set_ylabel('Oil Price (USD)', color=color2, fontsize=12)
ax2.tick_params(axis='y', labelcolor=color2)

plt.title('Unit Sales vs Oil Price Over Time', fontsize=14, fontweight='bold')
fig.legend(loc='upper right', bbox_to_anchor=(0.9, 0.9))
plt.tight_layout()
plt.show()

In [ ]:
# Correlation analysis
corr = df_oil['unit_sales'].corr(df_oil['dcoilwtico'])
print(f'Pearson correlation between sales and oil price: {corr:.4f}')

# Scatter plot
plt.figure(figsize=(10, 6))
plt.scatter(df_oil['dcoilwtico'], df_oil['unit_sales'], alpha=0.4, c='#3498db', edgecolors='black')
plt.xlabel('Oil Price (USD)', fontsize=12)
plt.ylabel('Unit Sales', fontsize=12)
plt.title('Sales vs Oil Price (Scatter Plot)', fontsize=14, fontweight='bold')

# Add trend line
z = np.polyfit(df_oil['dcoilwtico'].dropna(), df_oil.loc[df_oil['dcoilwtico'].notna(), 'unit_sales'], 1)
p = np.poly1d(z)
x_line = np.linspace(df_oil['dcoilwtico'].min(), df_oil['dcoilwtico'].max(), 100)
plt.plot(x_line, p(x_line), 'r--', linewidth=2, label=f'Trend (r={corr:.3f})')
plt.legend()
plt.grid(alpha=0.3)
plt.show()

In [ ]:
# Let's check if oil price CHANGES relate to sales
df_oil['oil_change'] = df_oil['dcoilwtico'].pct_change() * 100
df_oil['sales_change'] = df_oil['unit_sales'].pct_change() * 100

corr_change = df_oil['oil_change'].corr(df_oil['sales_change'])
print(f'Correlation between daily % change in oil and % change in sales: {corr_change:.4f}')

# Bucket oil price into quartiles and check sales
df_oil['oil_quartile'] = pd.qcut(df_oil['dcoilwtico'], 4, labels=['Q1 (Low)', 'Q2', 'Q3', 'Q4 (High)'])
oil_quartile_sales = df_oil.groupby('oil_quartile', observed=True)['unit_sales'].agg(['mean','std','count'])
print('\nSales by Oil Price Quartile:')
oil_quartile_sales

**Key Insight:** Oil price shows some relationship with sales, but it's not a strong linear correlation. The upward trend in sales during the 2014-2015 oil price crash suggests other factors are more dominant. However, oil price quartile analysis shows some variation.

---
## 5. Monthly & Yearly Patterns

In [ ]:
# Monthly pattern
df['month'] = df.index.month
month_names = ['Jan','Feb','Mar','Apr','May','Jun','Jul','Aug','Sep','Oct','Nov','Dec']
monthly_sales = df.groupby('month')['unit_sales'].mean()

plt.figure(figsize=(12, 5))
plt.bar(month_names, monthly_sales.values, color=['#e74c3c' if m in [12,1,2] else '#3498db' for m in range(1,13)],
        edgecolor='black')
plt.title('Average Sales by Month', fontsize=14, fontweight='bold')
plt.ylabel('Average Unit Sales')
for i, v in enumerate(monthly_sales.values):
    plt.text(i, v + 3, f'{v:.0f}', ha='center', fontweight='bold')
plt.show()

print(f'Highest month: {month_names[monthly_sales.argmax()]} ({monthly_sales.max():.0f})')
print(f'Lowest month: {monthly_sales.argmin()+1} ({monthly_sales.min():.0f})')

In [ ]:
# Yearly trend
df['year'] = df.index.year
yearly_sales = df.groupby('year')['unit_sales'].agg(['mean','sum'])
print('Yearly Sales:')
yearly_sales

In [ ]:
# Rolling 30-day average to see trend
df['rolling_30'] = df['unit_sales'].rolling(30).mean()

plt.figure(figsize=(14, 5))
plt.plot(df.index, df['unit_sales'], alpha=0.3, label='Daily Sales')
plt.plot(df.index, df['rolling_30'], color='#e74c3c', linewidth=2, label='30-Day Rolling Avg')
plt.title('Sales Trend (with 30-Day Rolling Average)', fontsize=14, fontweight='bold')
plt.ylabel('Unit Sales')
plt.legend()
plt.grid(alpha=0.3)
plt.show()

---
## 6. Summary of Findings

### Weekday Patterns
- **Weekends (Sat & Sun) have higher sales** than weekdays (+15-25% uplift)
- Wednesday tends to be the lowest sales day

### Holiday Impact
- **Holidays do impact sales**, though the effect varies by holiday type
- National holidays show a clearer pattern than local/regional ones
- **Christmas/Navidad** and **Black Friday** boost sales significantly
- Some holidays (like Dia de Difuntos) are associated with lower sales

### Oil Price
- The correlation between oil price and sales is **weak** (r ≈ -0.1 to -0.2)
- Sales increased during 2014-2016 when oil prices crashed, but this is likely coincidental with other factors
- Daily changes in oil price show no meaningful relationship with daily sales changes

### Seasonality
- Clear **monthly pattern**: December (Christmas) has the highest sales, February tends to be lowest
- Year-over-year: sales have been **growing** from 2013 to 2017
- The 30-day rolling average shows a generally **upward trend** with seasonal peaks